# 02 — MLP one-step predictor

`u_{n+1} = F(u_n)`, trained by mean squared error on every consecutive pair.

The one fact that governs the whole ladder: **the minimiser of MSE is the conditional mean**
`E[u_{n+1} | u_n]`. On the ODE the conditional is a point mass, so that mean *is* the flow map.
On the SDE it is not. See `l63.models.Predictor` for the derivation.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
KEY = 'ode'      # any of DATA
s = S['02_mlp_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('02_mlp_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

17,411 parameters, history 1, figures show seed 0

ruler             median   range over seeds
  horizon            336   [242–399] over 5 seeds
  spread               —   — over 0 seeds
  climate           1.90   [0.90–6.21] over 5 seeds
  climate_vs_truth    3.05   [1.45–10.01] over 5 seeds
  chaos             1.00   [0.91–1.01] over 5 seeds
  alive             1.00   [1.00–1.00] over 5 seeds
  lobe              0.64   [0.59–0.68] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  1.340e-02
  n=2  2.413e-02
  n=3  3.516e-02
  n=4  4.830e-02
  n=5  6.108e-02
  n=6  6.593e-02


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [3]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50051/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50051/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50051/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 